In [1]:
from datetime import time, timedelta

# COMPUTE NODE

In [2]:
# time format
# time is total time in milliseconds
# assuming IPT is Instrutions per second
import threading
import time

CLOCK_SPEED = 2

def getTime(instr,IPC):
    global CLOCK_SPEED
    time = (1./CLOCK_SPEED)*(float(instr)/(IPC))
    return round(time,2)*1000

class Process:
    instructions = 0
    size = 0
    name = ""
    def __init__(self,instructions,size,name):
        self.instructions = instructions
        self.size = size
        self.name = name

class Process_compute:
        process = None
        time_started = 0
        instructions_left = 0
        last_change_time = 0
        time_over = 0
        cpu_ipt_allocated = 0

        def __init__(self,process,ipt_alloc, time_now):
            self.process = process
            self.instructions_left = self.process.instructions
            self.time_started = time_now
            self.last_change_time = self.time_started
            self.cpu_ipt_allocated = ipt_alloc
            self.time_over = self.time_started + (self.instructions_left/float(self.cpu_ipt_allocated))*1000
        
        def showData(self):
            show = """
            ----------------------------------------------------------------
            Process Name: {}
            Total instructions in Process:{}
            Size of Process : {}
            Instructions left: {}
            Time started:{}
            Time it will get over:{}
            Last recorded time when Process resources were changed: {}
            cpu ipt allocated: {}
            ----------------------------------------------------------------
            """.format(self.process.name, self.process.instructions, self.process.size, self.instructions_left, self.time_started, self.time_over, self.last_change_time, self.cpu_ipt_allocated)

            # print(show)
            return show


class Compute_node:
    
    name = ""
    IPT = 0
    RAM = 0
    thresh = 5
    num_process = 0

    processes = []
    message_queue = []
    show_queue = []
    process_logs = []

    time_created = 0
    simulation_time = 0

    def __init__(self,ipt,ram, thresh, time_now, simulation_time,name="compute_node"):
        self.processes = []
        self.show_queue = []
        self.message_queue = []
        self.process_logs = []
        self.IPT = ipt
        self.RAM = ram
        self.thresh = thresh
        self.time_created = time_now
        self.simulation_time = simulation_time
        thread = threading.Thread(target=self.getUpdates)
        thread.start()
        self.name = name

    def getUpdates(self):
        time_now = 0
        time.sleep(0.3)
        while(time_now < self.time_created + self.simulation_time):
            time_now += 500
            self.update_processes(time_now)
   

    def show_stats(self,time=0):
        self.show_queue.append(time)
            
# show function is called in update_processes function
    def show(self,time):
        log = """
        ============================================================
        Compute node name - {}
        ------------------------------------------------------------
        time of message : {}
        number of process running: {}
        Current Processes info
        """.format(self.name,time,self.num_process)

        for process in self.processes:
            log += "\n"
            log += process.showData()
            log += "\n"

        log+="\n\t============================================================\n\n"
        
        self.process_logs.append(log)

    def update_processes(self, time_now):

        # removal of processes already completed
        remove_list = []
        for i in self.processes:
            if i.time_over < time_now:
                remove_list.append(i)

        if len(remove_list) > 0:
            for i in remove_list:
                self.processes.remove(i)
            

        # addition of new processes
        add_list = []

        if self.num_process < self.thresh or len(remove_list)>0:
        
            while(self.num_process <= self.thresh and self.message_queue != [] and (self.num_process + len(add_list) - len(remove_list)< self.thresh)):
                if self.message_queue[0][0] <= time_now:
                    add_list.append(self.message_queue[0][1])
                    self.message_queue.pop(0)
                else:
                    break

        # updating the info of already existing processes

        self.num_process = self.num_process + len(add_list) - len(remove_list)

        for i in self.processes:
            i.instructions_left = i.instructions_left - i.cpu_ipt_allocated*(float(time_now - i.last_change_time)/1000)
            # print("!! "+  str(i.cpu_ipt_allocated*(float(time_now - i.last_change_time)/1000) ))
            i.cpu_ipt_allocated = float(self.IPT)/self.num_process
            timedelta = i.instructions_left/float(i.cpu_ipt_allocated)  # assuming that timedelta is in seconds
            i.time_over = time_now + timedelta*1000                     # conversion from seconds to milliseconds
            i.last_change_time = time_now

        # adding processes one by one
        for add_p in add_list:
            new_process = Process_compute(add_p,float(self.IPT)/self.num_process, time_now)
            self.processes.append(new_process)

        # showing Data
       
        if self.show_queue != [] and self.show_queue[0] <= time_now:
            self.show_queue.pop(0)
            self.show(time_now)
            # for process in self.processes:
            #     process.showData()
            

    def when_free(self,time_now):
        self.update_processes()
        if self.num_process < self.thresh:
            # returning a time less than current time
            return time_now - 100                   
        else:
            min_t = 0
            for i in self.processes:
                min_t = min(min_t, i.time_over)

            return min_t

    def add_process(self,process,time):
        self.message_queue.append((time,process))

     

# Some processes

In [ ]:
#setting up a test case

# node = Compute_node(20,100,5)
process1 = Process(100,5,"process1")
process2 = Process(100,5,"process2")
process3 = Process(100,5,"process3")
process4 = Process(100,5,"process4")
process5 = Process(100,5,"process5")
process6 = Process(100,5,"process6")
process7 = Process(100,5,"process7")
process8 = Process(100,5,"process8")
process9 = Process(100,5,"process9")

# TEST CASE 1

In [ ]:
# import asyncio

# async def function_asyc(update_process,simul_time):
#     time_now = 0
#     while(time_now < simul_time):
#         time_now += 500
#         update_process(time_now)
   


simulation_time = 30 # in seconds
print("Start")
node = Compute_node(20,100,5,0,simulation_time*1000)  
# asyncio.create_task(function_asyc(node.update_processes,simulation_time*1000))
print("Compute node initialized")
time1 = 1  # in seconds
node.add_process(process1,time1*1000)
node.show_stats(1500)
node.show_stats
time2 = 4
node.add_process(process2,time2*1000)
node.show_stats(4500)
node.show_stats(9000)
node.show_stats(16000)
node.show_stats(20000)


    


# loop = asyncio.get_event_loop()

# loop.run_until_complete(asyncio.gather(main(),function_asyc(node.update_processes,simulation_time*1000)))

# loop.close()

# TEST CASE 2

In [ ]:
import asyncio

async def function_asyc(update_process,simul_time):
    time_now = 0
    while(time_now < simul_time):
        time_now += 500
        update_process(time_now)
   


simulation_time = 60
print("Start")
node = Compute_node(20,100,5,0,simulation_time*1000) 
print("Compute node initialized")
time1 = 1  # in seconds
node.add_process(process1,0.5*1000)
node.add_process(process2,1*1000)
node.add_process(process3, 1.5*1000)
node.add_process(process4, 2*1000)
node.add_process(process5, 2.5*1000)
node.show_stats(2500)
time2 = 4
node.add_process(process6, 4*1000)
node.add_process(process7, 5*1000)
node.show_stats(4500)
node.show_stats(5500)
node.show_stats(23000)
node.show_stats(25000)
node.show_stats(30000)
node.show_stats(35000)
node.show_stats(40000)

# Setting up a test network

In [3]:
import random

class Link:
    src = None
    dst = None
    bandwidth = 0
    prop_delay = 0

    def __init__(self,src,dst,band,prop):
        self.src = src
        self.dst = dst
        self.bandwidth = band
        self.prop_delay = prop

    def get_trans_time(self,size):
        return size/self.bandwidth
    
    def send_msg(self,process,time_now):
        self.dst.add_process(process, time_now + self.prop_delay + self.get_trans_time(process.size))


class Sensors:
    name = ""

    links = []
    def __init__(self, name):
        self.name = name
    num_process_sent = 1

    def send(self,time_now,name="p"):
        p = Process( 100,50, self.name + "_" + str(self.num_process_sent) + "_" + name )
        x = random.randint(0,len(self.links)-1)
        self.links[x].send_msg(p,time_now)
        self.num_process_sent +=1


In [4]:
# creating edge devices
edge1 = Compute_node(20,100,5,0,100*1000,"c1")
edge2 = Compute_node(20,100,5,0,100*1000,"c2")
edge3 = Compute_node(20,100,5,0,100*1000,"c3")
compute_devices = [edge1,edge2,edge3]

# creating sensors
s1 = Sensors("s1")
s2 = Sensors("s2")
s3 = Sensors("s3")
s4 = Sensors("s4")
s5 = Sensors("s5")

sensors_list = [s1,s2,s3,s4,s5]

# creating links
for s in sensors_list:
    for c in compute_devices:
        link = Link(s,c,10,1)
        s.links.append(link)

# running the simulation
for time in range(1000,50000,500):
    for cn in compute_devices:
        cn.show_stats(time)
    if(time%3000==0):
        x = random.randint(0,len(sensors_list)-1)
        sensors_list[x].send(time)


In [5]:
logs = []
for i in compute_devices:
    logs.append(i.process_logs)


for i in range(len(logs[0])):
    for j in range(len(logs)):
        print(logs[j][i])


        Compute node name - c1
        ------------------------------------------------------------
        time of message : 1000
        number of process running: 0
        Current Processes info
        



        Compute node name - c2
        ------------------------------------------------------------
        time of message : 1000
        number of process running: 0
        Current Processes info
        



        Compute node name - c3
        ------------------------------------------------------------
        time of message : 1000
        number of process running: 0
        Current Processes info
        



        Compute node name - c1
        ------------------------------------------------------------
        time of message : 1500
        number of process running: 0
        Current Processes info
        



        Compute node name - c2
        ------------------------------------------------------------
        time of message : 1500
        number of proces

In [ ]:
import threading
import time

class ab:
    name = ""
    def __init__(self,name):
        self.name = name
        thread = threading.Thread(target=self.a)
        thread.start()
    def a(self):
        print(self.name)
        time.sleep(0.1)
        for i in range(1,15,3):
            print(i)

# def b():
#     time.sleep(0.1)
#     for i in range(2,15,3):
#         print(i)

# def c():
#     time.sleep(0.1)
#     for i in range(3,15,3):
#         print(i)

a1 = ab("a1")
a2 = ab("a2")
a3 = ab("a3")

functions =[a1,a2,a3]

# for i in functions:
    # thread = threading.Thread(target=i.a)
    # thread.start()

# thread = threading.Thread(target=b)
# thread.start()

# thread = threading.Thread(target=c)
# thread.start()